# 第 1 章：深度学习基础

对应本地《深度强化学习》drl_v1.pdf 的第一章，书内第 3–14 页（PDF 第 13–24 页）。这章是进入强化学习前的工具复习：线性模型、神经网络、梯度下降与反向传播。

你已学过回归、MLP/CNN 和 PyTorch 自动求导。本笔记重在把熟悉的内容对齐到书中的符号，不再搭建训练项目。


## 本章主线

1. **1.1 线性模型**：输入如何产生预测？回归、二分类和多分类分别怎样定义输出与损失？
2. **1.2 神经网络**：线性模型不够用时，怎样组合多层变换？
3. **1.3 反向传播与梯度下降**：知道损失后，怎样改进模型参数？

读公式时始终辨认四件事：输入、输出、参数、优化目标。


## 1.1 线性模型（书内第 3–8 页）

设输入特征是列向量 $\boldsymbol{x}\in\mathbb{R}^{d}$，最简单的模型给出一个标量预测：

$$
\hat y=\boldsymbol{x}^{\top}\boldsymbol{w}+b
$$

$\boldsymbol{w}\in\mathbb{R}^{d}$ 是权重，$b\in\mathbb{R}$ 是偏置。以房价为例，面积、房龄等组成输入 $\boldsymbol{x}$，模型用它们的加权和估计价格。输入是某个样本已知的信息；参数是训练中需要学到的量。书中统一把向量写成列向量，后面检查矩阵维度要沿用这一约定。


### 回归：损失与优化目标

对 $n$ 个样本 $(\boldsymbol{x}_i,y_i)$，最小二乘回归用平均平方误差衡量预测与真实值的差别：

$$
L(\boldsymbol{w},b)
=\frac{1}{2n}\sum_{i=1}^{n}(\hat y_i-y_i)^2
$$

系数 $\frac12$ 不改变最优点，只让求导更简洁。书中还加入正则项 $R(\boldsymbol{w})$ 约束参数，因此要解：

$$
(\boldsymbol{w}^{\star},b^{\star})
=\underset{\boldsymbol{w},b}{\arg\min}
\big[L(\boldsymbol{w},b)+R(\boldsymbol{w})\big]
$$

**min 给出最小函数值；argmin 给出使函数最小的参数。**以后看到训练目标，先确认究竟在优化哪些变量。


### 二分类：线性输出后接 Sigmoid

若标签只有 $0$ 和 $1$，先算 $z=\boldsymbol{x}^{\top}\boldsymbol{w}+b$，再把它压到 $(0,1)$：

$$
p=\sigma(z)=\frac{1}{1+e^{-z}}
$$

$p$ 是模型对类别 1 的预测概率。一个样本的二元交叉熵为：

$$
\ell(y,p)=-y\log p-(1-y)\log(1-p)
$$

若真实标签为 $1$，预测 $p$ 很小会受较大惩罚；标签为 $0$ 时则相反。“逻辑斯蒂回归”虽然名字含回归，本章用它做二分类。模型输出能否与真实频率匹配，还取决于训练和校准。


### 多分类：Softmax 给每一类一个预测概率

若有 $k$ 个互斥类别，先计算 $k$ 个分数：

$$
\boldsymbol{z}=W\boldsymbol{x}+\boldsymbol{b},
\qquad
W\in\mathbb{R}^{k\times d},
\quad \boldsymbol{z},\boldsymbol{b}\in\mathbb{R}^{k}
$$

Softmax 把分数变成总和为 1 的向量：

$$
p_j=\frac{e^{z_j}}{\sum_{\ell=1}^{k}e^{z_\ell}}
$$

书中用 MNIST 举例：$28\times28$ 图像展平成 $d=784$ 维输入，数字 0–9 对应 $k=10$ 类。真实标签可写成 one-hot 向量 $\boldsymbol{y}$，损失为交叉熵 $-\sum_{j=1}^{k}y_j\log p_j$。

| 任务 | 输出 | 常见输出层 | 本章的损失 |
| --- | --- | --- | --- |
| 回归 | 连续数值 | 线性输出 | 平方误差 |
| 二分类 | 类别 1 的预测概率 | Sigmoid | 二元交叉熵 |
| 多分类 | $k$ 类预测分布 | Softmax | 多类交叉熵 |


### 三种任务，共同的训练骨架

三种模型都遵循：**准备带标签数据 → 定义模型与损失 → 用优化算法调整参数**。你做过的回归、MNIST 分类与 PyTorch 训练循环，都是这条骨架的实例。

后面学深度强化学习时，网络依然需要预测和更新参数，但训练目标往往来自与环境的交互。这是本书先回顾监督学习模型的原因。


## 1.2 神经网络（书内第 9–11 页）

全连接层先做线性变换，再接激活函数：

$$
\boldsymbol{z}^{(\ell)}
=W^{(\ell)}\boldsymbol{x}^{(\ell-1)}+\boldsymbol{b}^{(\ell)},
\qquad
\boldsymbol{x}^{(\ell)}=\sigma_{\ell}(\boldsymbol{z}^{(\ell)})
$$

输入维度为 $d_{\mathrm{in}}$、输出维度为 $d_{\mathrm{out}}$ 时，权重矩阵形状为 $d_{\mathrm{out}}\times d_{\mathrm{in}}$。多层相接得到多层感知器（MLP），各层有自己的参数。

隐藏层为什么需要 ReLU 等**非线性**激活？如果只有线性变换与偏置，多层仍可合并成一个线性变换，堆层并不会真正增加表达能力。输出层的选择则取决于任务：回归可线性输出，二分类用 Sigmoid，多分类用 Softmax。


### CNN 在本章的角色

书中只简要回顾卷积神经网络（CNN）：图片以矩阵或多维张量输入，卷积层提取特征，最后将特征交给全连接层完成预测；这里不展开卷积核、padding 等细节。

你已学过 CNN，可把这一节看成角色定位：面对图像输入，CNN 能充当特征提取器。本章不需要另写一个 CNN。


## 1.3 梯度下降与反向传播（书内第 12–14 页）

把全部参数统称为 $\theta$，训练目标是减小损失 $L(\theta)$。梯度指向损失局部增大的方向，因此梯度下降沿反方向更新：

$$
\theta_{\mathrm{new}}
=\theta_{\mathrm{old}}-\alpha\nabla_{\theta}L(\theta_{\mathrm{old}})
$$

$\alpha>0$ 是学习率。损失是标量；关于某个参数的梯度与该参数**形状相同**。例如 $W$ 是矩阵，$\nabla_W L$ 也是同形状矩阵。

书中用全数据计算梯度说明 GD，用随机抽一个样本说明 SGD。你在 PyTorch 中常见的小批量更新，每步用一小批样本估计梯度。三者都利用梯度改进参数。


### 反向传播负责算梯度

前向计算逐层把输入变成预测，再得到损失；反向传播从损失出发，利用链式法则逐层把梯度传回去。最简单地，若 $x\rightarrow h\rightarrow L$，则：

$$
\frac{\partial L}{\partial x}
=\frac{\partial L}{\partial h}
\frac{\partial h}{\partial x}
$$

对每一层，损失对**本层参数**的梯度用于更新参数；损失对**本层输入**的梯度继续传给前一层。PyTorch 的 loss.backward() 负责计算梯度，optimizer.step() 使用梯度更新参数。

**反向传播是算梯度的方法；梯度下降是用梯度更新参数的方法。**


## 手算一个最小更新

一个样本：$x=2$，目标 $y=3$；模型 $\hat y=wx+b$，初始 $w=1,b=0$。取单样本损失 $L=\frac12(\hat y-y)^2$，学习率 $\alpha=0.1$。

1. 前向：$\hat y=2$，所以 $L=0.5$。
2. 梯度：$\frac{\partial L}{\partial w}=(\hat y-y)x=-2$，$\frac{\partial L}{\partial b}=\hat y-y=-1$。
3. 更新：$w_{\mathrm{new}}=1.2$，$b_{\mathrm{new}}=0.1$。
4. 再预测：$\hat y_{\mathrm{new}}=1.2\times2+0.1=2.5$；新损失为 $0.125$。

这把“模型 → 损失 → 梯度 → 更新”串成了一条线。多层网络只是求梯度的路径更长。


## 本章小结与自检

第一章统一了后文会用到的语言：模型怎样从输入得到预测、怎样用损失定义目标、神经网络怎样增加表达能力、梯度怎样把误差传回参数。本章尚未介绍强化学习算法；概率基础和 MDP 分别在第二、三章。

请独立回答：

1. 输入为 4 维、类别数为 3 时，线性 Softmax 分类器的权重和输出各是什么形状？
2. 为什么多层网络的隐藏层通常需要非线性激活？
3. min 与 argmin 分别返回什么？
4. 矩阵参数的梯度应是什么形状？
5. loss.backward() 和 optimizer.step() 各负责什么？
6. 不看上面的解答，重算“手算一个最小更新”。

前五题能解释清楚、第六题能复算，就完成本章复习。


## 阅读定位与保密

本笔记对应本地 drl_v1.pdf 第一章“深度学习基础”：1.1 线性模型（书内第 3–8 页）、1.2 神经网络（第 9–11 页）、1.3 反向传播和梯度下降（第 12–14 页）。内容以自己的话组织，未复制原书的大段文字。PDF 保留在本机，并已被 Git 忽略。
